In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from typing import TypedDict
from dotenv import load_dotenv
import os
print(os.getenv("HUGGING_FACE_API_KEY"))

In [35]:
load_dotenv()  

True

In [44]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=os.getenv("HUGGING_FACE_API_KEY")
)
model = ChatHuggingFace(llm=llm)

In [45]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [46]:
def create_outline(state: BlogState) -> BlogState:

    title=state['title']
    prompt = f"Generate a detailed outline for a blog on the topic - {title}"
    outline = model.invoke(prompt).content

    #upadte state
    state['outline'] = outline

    return state

In [47]:
def create_blog(state:BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f"Write a detailed blog on the tile - {title} using the following outline \n{outline}"

    content = model.invoke(prompt).content
     
    state['content'] =  content

    return state


In [49]:
graph = StateGraph(BlogState)

#nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

#edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)


#execute 

workflow = graph.compile()


In [53]:
intial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(intial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': '**Title:** "The Rise of AI in India: Unlocking Opportunities and Transforming Industries"\n\n**I. Introduction**\n\n* Brief overview of the growing importance of Artificial Intelligence (AI) globally and in India\n* Thesis statement: The rise of AI in India is expected to revolutionize various sectors, drive innovation, and propel the country\'s economic growth.\n\n**II. Current AI Landscape in India**\n\n* Overview of the current state of AI adoption in India\n* Statistics on AI investments, research, and development in India\n* Examples of successful AI startups and companies in India\n\n**III. AI Applications in India\'s Key Industries**\n\n* **Healthcare:** AI in medical diagnosis, personalized medicine, and healthcare analytics\n* **Finance:** AI in risk management, predictive analytics, and customer service\n* **Education:** AI in adaptive learning, intelligent tutoring systems, and educational analytics\n* **E-commerce:** AI in person

In [54]:
print(final_state['outline'])

**Title:** "The Rise of AI in India: Unlocking Opportunities and Transforming Industries"

**I. Introduction**

* Brief overview of the growing importance of Artificial Intelligence (AI) globally and in India
* Thesis statement: The rise of AI in India is expected to revolutionize various sectors, drive innovation, and propel the country's economic growth.

**II. Current AI Landscape in India**

* Overview of the current state of AI adoption in India
* Statistics on AI investments, research, and development in India
* Examples of successful AI startups and companies in India

**III. AI Applications in India's Key Industries**

* **Healthcare:** AI in medical diagnosis, personalized medicine, and healthcare analytics
* **Finance:** AI in risk management, predictive analytics, and customer service
* **Education:** AI in adaptive learning, intelligent tutoring systems, and educational analytics
* **E-commerce:** AI in personalization, customer service, and supply chain management
* **Manu